![club_data_model2.jpg](./club_data_model2.jpg "club_data_model2.jpg")

In [0]:
#load facilities
from pyspark.sql.types import StructField, StructType,IntegerType,StringType,FloatType

facilities_schema  = StructType([
     StructField("facid",StringType(), False),
     StructField("fac_name",StringType(), False),
     StructField("membercost",FloatType(), False),
     StructField("guestcost",FloatType(), False),
     StructField("initialoutlay",FloatType(), False),
     StructField("monthlymaintenance",IntegerType(), False)
 ]);


facilities_df = spark.read.format('csv').\
    schema(facilities_schema).options(header =True)\
           .load('/Volumes/dev/spark_db/datasets/spark_programming/data/facilities.csv')


In [0]:
facilities_df.display()

In [0]:
#load booking df
from pyspark.sql.types import TimestampType

bookings_schema  = StructType([
     StructField("bookid",IntegerType(), False),
     StructField("facid",StringType(), True),
     StructField("memid",IntegerType(), True),
     StructField("starttime",TimestampType(), True),
     StructField("slots",FloatType(), True),
 ]);


bookings_df = spark.read.format('csv').\
    schema(bookings_schema).option("header" , True)\
        .option("timestampFormat","yyyy-MM-dd'T'HH:mm:ss.SSSXX")\
           .load('/Volumes/dev/spark_db/datasets/spark_programming/data/bookings.csv')


In [0]:
bookings_df.display()

In [0]:
print("Session timezone:", spark.conf.get('spark.sql.session.timeZone'))

In [0]:
spark.conf.set('spark.sql.session.timeZone' , "UTC")

In [0]:
#load members data
from pyspark.sql.types import DateType

members_schema  = StructType([
     StructField("memid",IntegerType(), False),
     StructField("surname",StringType(), True),
     StructField("firstname",StringType(), True),
     StructField("address",StringType(), True),
     StructField("zipcode",StringType(), True),
     StructField("telephone",StringType(), True),
     StructField("recommendedby",StringType(), True),
     StructField("joindate",DateType(), True),
 ]);


members_df = spark.read.format('csv').\
    schema(members_schema).option("header" , True)\
        .option("dateFormat","yyyy-MM-dd'T'HH:mm:ss.SSSXX")\
           .load('/Volumes/dev/spark_db/datasets/spark_programming/data/members.csv')


In [0]:
members_df.display()

%md
Q1. Prepare a facility bookings reporting dataset as the following.
```
member_id | first_name | last_name | facility_id | slots | start_time
---------------------------------------------------------------------------
```
The report must meet the following criteria.
1. Facility bookings made by a person whose last name is Smith
2. He has booked more than 5 slots in a single booking
3. Report should be sorted by first name of the member in ascending order and number of slots in descending order


saving df in table

In [0]:
facilities_df.write.mode('overwrite').option('overwriteSchema', 'True').saveAsTable('dev.spark_db.facilities');

bookings_df.writeTo('dev.spark_db.bookings').createOrReplace();

members_df.writeTo('dev.spark_db.members').createOrReplace()

SQL method

In [0]:
%sql
SELECT b.bookid, b.facid, m.surname,m.firstname,b.slots
FROM dev.spark_db.bookings  b
INNER JOIN dev.spark_db.members  m
ON b.memid = m.memid
WHERE m.surname like '%Smith%'
AND b.slots >5
ORDER BY b.slots DESC

In [0]:
%sql
WITH slots_greater_than_5_booking AS 
(SELECT b.bookid, b.facid, m.surname,m.firstname,b.slots
FROM dev.spark_db.bookings  b
INNER JOIN dev.spark_db.members  m
ON b.memid = m.memid
WHERE m.surname like '%Smith%'
AND b.slots >5
ORDER BY b.slots DESC)

SELECT facid,SUM(slots) AS slot_count
FROM slots_greater_than_5_booking
GROUP BY facid
ORDER BY slot_count DESC

In [0]:
%sql
DESCRIBE EXTENDED dev.spark_db.members

Using pyspark

In [0]:
bookings_df = spark.read.table("dev.spark_db.bookings");
members_df =spark.read.table("dev.spark_db.members");

facilties_df =spark.read.table("dev.spark_db.facilties");


In [0]:
member_booking_join = bookings_df.join(members_df, on= 'memid', how ='inner')

In [0]:
from pyspark.sql.functions import col, contains,desc,sum

filter_smith_slots = (member_booking_join.filter(
        (col('surname').contains('Smith')) & (col('slots')>5))
        .select('bookid','facid','surname','firstname','slots')
        .orderBy(desc('slots'))
);


#groupby
fac_booking_groupby_sum = filter_smith_slots.groupBy('facid').agg(sum('slots').alias('booking_count')).orderBy(col('booking_count').desc())


In [0]:
filter_smith_slots.display()


#### Outer Join


Q1. List all bookings made by a person named Darren Smith as the following.
```
member_id | first_name | last_name | address | facility_id | slots
---------------------------------------------------------------------
```

Ensure the following
1. Show the details of all persons named Darren Smith even if they have not made any bookings
2. Sort the result by number of slots (highers first)
3. List the person with no bookings at the top

In [0]:
bookings_df.display();

members_df.display();

facilities_df.display()

```
member_id | first_name | last_name | address | facility_id | slots
---------------------------------------------------------------------
```

In [0]:
%sql
 SELECT m.memid,m.firstname, m.surname,m.address,b.facid,b.slots
FROM dev.spark_db.members m
LEFT JOIN dev.spark_db.bookings b
ON m.memid = b.memid
WHERE m.firstname ILIKE '%Darren%' AND m.surname ILIKE '%Smith%'
ORDER BY memid DESC, slots DESC

In [0]:
#python way

bookings_df = spark.read.table("dev.spark_db.bookings");
members_df =spark.read.table("dev.spark_db.members");

facilties_df =spark.read.table("dev.spark_db.facilties");

In [0]:
from pyspark.sql.functions import col,lower

bookings_of_darren_smith = (
                    members_df.join
                        (bookings_df, on="memid",how="left")
                     .select(
                            "memid", "firstname", "surname","address","facid","slots"
                            )
                        .filter(
                                (lower(col("firstname")).contains("darren")) & 
                                (lower(col("surname")).contains("smith"))
                        )
                        .orderBy(col('memid').desc(),col('slots').desc())                       
)

In [0]:
bookings_of_darren_smith.display()